# Classification DPE — source commune

Le travail historique sur le Rhône est conservé intégralement dans `archives/2_classification_dpe_rhone_historique.ipynb`. Ses scores ne sont pas ceux du CSV commun. Ce notebook utilise les mêmes sept caractéristiques de logement que la régression, sans coût ni consommation. Un seul Random Forest à profondeur limitée est évalué, sans recherche longue. Les performances de F et G doivent être lues séparément.

Les scores historiques sont liés à la sauvegarde antérieure aux corrections géographiques ; ils ne valident pas le nouveau CSV. La comparaison future utilise une validation séparée.


In [1]:
from pathlib import Path
import os
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "MLmodels").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Ouvrez le notebook depuis le dépôt App_Enedis.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Racine du projet :", ROOT)

import json
import joblib
import numpy as np
from src.ml_workflow import (load_data, split_classification, train_classification,
                             CLASS_MODEL, CLASS_REPORT, fingerprint)
RUN_TRAINING = False  # Activer uniquement pour refaire l'entraînement.


Racine du projet : C:\Users\mouni\Documents\App_Enedis


In [2]:
from src.ml_workflow import reference_data_path
if RUN_TRAINING:
    raise RuntimeError("Utiliser la comparaison limitée entraînement/validation. Aucun entraînement automatique ici.")
DATA_FOR_EVALUATION = reference_data_path()
print("Référence historique avant corrections géographiques :", DATA_FOR_EVALUATION)
df = load_data(DATA_FOR_EVALUATION)
print("Dimensions :", df.shape)
print(df.etiquette_dpe.value_counts().sort_index())


Référence historique avant corrections géographiques : C:\Users\mouni\Documents\App_Enedis\output\backups\cleaned_data_4ac7b5d2b0a4f3ba.csv
Dimensions : (616057, 16)
etiquette_dpe
A    156169
B     68006
C    187936
D    103758
E     61514
F     24084
G     14590
Name: count, dtype: int64


In [3]:
if RUN_TRAINING:
    pipeline, infos = train_classification(df)
else:
    pipeline = joblib.load(CLASS_MODEL)  # pipeline_classification_common.pkl
    infos = json.loads(CLASS_REPORT.read_text(encoding='utf-8'))
if infos['data_sha256'] != fingerprint(DATA_FOR_EVALUATION):
    raise ValueError('Le CSV a changé : réentraîner avant de publier des résultats.')
X_train, X_test, y_train, y_test = split_classification(df)
y_pred_rf = pipeline.predict(X_test)


In [4]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
print('Accuracy :', accuracy_score(y_test, y_pred_rf))
print('F1 macro :', f1_score(y_test, y_pred_rf, average='macro'))
print(classification_report(y_test, y_pred_rf, digits=3, zero_division=0))
np.testing.assert_allclose(accuracy_score(y_test, y_pred_rf), infos['accuracy'])
np.testing.assert_allclose(f1_score(y_test, y_pred_rf, average='macro'), infos['f1_macro'])


Accuracy : 0.5795809931933037
F1 macro : 0.4759462742330875
              precision    recall  f1-score   support

           A      0.909     0.885     0.897     46851
           B      0.485     0.610     0.541     20402
           C      0.835     0.487     0.615     56381
           D      0.433     0.474     0.453     31128
           E      0.312     0.328     0.319     18454
           F      0.181     0.285     0.221      7225
           G      0.182     0.659     0.286      4377

    accuracy                          0.580    184818
   macro avg      0.477     0.533     0.476    184818
weighted avg      0.654     0.580     0.598    184818



In [5]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, labels=list('ABCDEFG'))
plt.title('DPE — source commune, test stratifié de 30 %')
plt.show()


In [6]:
modele_recharge = joblib.load(CLASS_MODEL)
np.testing.assert_array_equal(pipeline.predict(X_test.head(20)), modele_recharge.predict(X_test.head(20)))
print('Prédictions identiques après rechargement.')


Prédictions identiques après rechargement.


Les paramètres sont fixés avant l'évaluation ; aucun réglage n'est sélectionné sur le test. L'évaluation reste interne au CSV, sans validation sur une autre période ou un autre territoire. Le DPE prédit est une estimation statistique, pas un diagnostic réglementaire. Les modèles historiques du Rhône restent dans MLmodels sous leurs noms initiaux.
